# Final private inference — deterministic A100

Runs pinned Qwen2.5-3B + frozen R3 R2-continuation LoRA entirely locally with vLLM batch invariance and synchronous scheduling. The command is resume-safe and writes exact submission/audit artifacts to Drive.

In [ ]:
# Cell 1 — Clone the exact deterministic code revision and install. Restart once afterward.
import subprocess,sys
from pathlib import Path
CODE_COMMIT="2254a166b1f607a9fb7a7d55b1136c0238aa21a1"
REPO=Path("/content/qwen-math-final-2026")
if not REPO.exists():
    subprocess.run(["git","clone","https://github.com/jhparktime/qwen-math-final-2026.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin",CODE_COMMIT],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",CODE_COMMIT],check=True)
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT,(actual,CODE_COMMIT)
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir","-r",str(REPO/"requirements-colab.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-q","-y","torchcodec"],check=False)
print("[CODE COMMIT]",actual);print("[SETUP] complete; restart runtime once, then continue at Cell 2")

In [ ]:
# Cell 2 — After restart, restore the repository path and mount Drive.
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026");CODE_COMMIT="2254a166b1f607a9fb7a7d55b1136c0238aa21a1"
assert REPO.exists(),REPO
import subprocess
actual=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip();assert actual==CODE_COMMIT,(actual,CODE_COMMIT)
from google.colab import drive
drive.mount("/content/drive")
print("[REPO]",REPO);print("[CODE COMMIT]",actual)

In [ ]:
# Cell 3 — One-time base-model cache preparation. No test data is read or sent.
import os
from huggingface_hub import snapshot_download
os.environ.pop("HF_HUB_OFFLINE",None)
os.environ.pop("TRANSFORMERS_OFFLINE",None)
BASE_MODEL="Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION="aa8e72537993ba99e69dfaafa59ed015b17504d1"
model_cache=snapshot_download(repo_id=BASE_MODEL,revision=MODEL_REVISION)
print("[BASE MODEL CACHED]",model_cache)

In [ ]:
# Cell 4 — Set only these three Drive paths.
from pathlib import Path
REPO=Path("/content/qwen-math-final-2026")
INPUT_PATH=Path("/content/drive/MyDrive/test_submission.csv")
ADAPTER_PATH=Path("/content/drive/MyDrive/2026소중한챌린지/runs/RFT-0008D-r3mix-r2continue-r16-a100/adapter_final")
OUTPUT_DIR=Path("/content/drive/MyDrive/2026소중한챌린지/runs/FINAL-0004-r3-r2continue-sc16-pal3-deterministic")
assert INPUT_PATH.exists(),INPUT_PATH
assert (ADAPTER_PATH/"adapter_config.json").exists(),ADAPTER_PATH
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print(INPUT_PATH,ADAPTER_PATH,OUTPUT_DIR,sep="\n")

In [ ]:
# Cell 5 — Offline, resume-safe final inference with live logs.
import os,subprocess,sys
env=os.environ.copy();env["PYTHONPATH"]=str(REPO);env["HF_HUB_OFFLINE"]="1";env["TRANSFORMERS_OFFLINE"]="1"
command=[sys.executable,str(REPO/"inference/final_inference.py"),"--input",str(INPUT_PATH),"--adapter",str(ADAPTER_PATH),"--output-dir",str(OUTPUT_DIR)]
process=subprocess.Popen(command,cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
for line in process.stdout: print(line,end="",flush=True)
exit_code=process.wait()
if exit_code: raise RuntimeError(f"Inference failed with exit code {exit_code}")

In [ ]:
# Cell 6 — Exact submission validation.
SUBMISSION=OUTPUT_DIR/"submissions/submission.csv"
subprocess.run([sys.executable,str(REPO/"scripts/validate_submission.py"),"--input",str(INPUT_PATH),"--submission",str(SUBMISSION),"--expected-rows","2000"],cwd=REPO,check=True)
print("[FINAL SUBMISSION]",SUBMISSION)

In [ ]:
# Final cell — Optional GPU runtime release.
DISCONNECT_GPU_RUNTIME=False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained")